In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import os
import re
import string
import shutil
import tensorflow as tf
from tensorflow.keras import layers, losses, callbacks, Sequential

In [ ]:
url = "https://ai.stanford.edu/~amaas/data/sentiment/aclImdb_v1.tar.gz"
dataset = tf.keras.utils.get_file(
    "aclImdb_v1" , url,
    untar=True, cache_dir='.',
    cache_subdir='.'
)
dataset_dir = os.path.join(os.path.dirname(dataset), 'aclImdb_v1')

In [ ]:
os.listdir(dataset_dir)

In [ ]:
train_dir="aclImdb_v1/aclImdb/train"
os.listdir(train_dir)

In [ ]:
batch_size = 32
seed = 42

raw_train_ds = tf.keras.utils.text_dataset_from_directory(
    train_dir,
    batch_size=batch_size,
    validation_split=0.2,
    subset='training',
    seed=seed
)
raw_val_ds = tf.keras.utils.text_dataset_from_directory(
    train_dir,
    batch_size=batch_size,
    validation_split=0.2,
    subset='validation',
    seed=seed
)
raw_test_ds = tf.keras.utils.text_dataset_from_directory(
    "aclImdb_v1/aclImdb/test",
    batch_size=batch_size,
)

In [ ]:
for x in iter(raw_test_ds):
    print(x)
    break

In [ ]:
def custom_standardization(input_data):
    l_case = tf.strings.lower(input_data)
    stp_html = tf.strings.regex_replace(
        l_case, '<br/>', ' '
    )
    return tf.strings.regex_replace(
        stp_html,
        f'[{re.escape(string.punctuation)}]',
        ''
    )

def vectorize_text(text, label):
    text = tf.expand_dims(text, -1)
    return vectorize_layer(text), label

In [ ]:
max_features = 10000
seq_len = 250

vectorize_layer = layers.TextVectorization(
    standardize=custom_standardization,
    max_tokens=max_features,
    output_mode='int',
    output_sequence_length=seq_len 
)

In [ ]:
train_text = raw_train_ds.map(lambda x, y: x)
vectorize_layer.adapt(train_text)

In [ ]:

AUTOTUNE = tf.data.AUTOTUNE

train_ds = raw_train_ds.map(vectorize_text).cache().prefetch(buffer_size=AUTOTUNE)
val_ds = raw_val_ds.map(vectorize_text).cache().prefetch(buffer_size=AUTOTUNE)
test_ds = raw_test_ds.map(vectorize_text).cache().prefetch(buffer_size=AUTOTUNE)

In [ ]:
emb_dims = 128

model = Sequential([
    layers.Embedding(
        len(vectorize_layer.get_vocabulary()), 64, mask_zero=True),
    layers.Bidirectional(
        layers.LSTM(64, return_sequences=True)
    ),
    layers.Bidirectional(
        layers.LSTM(32),
    ),
    layers.Dense(64, activation='relu'),
    layers.Dense(1),
])

model.compile(
    optimizer='adam',
    loss=losses.BinaryCrossentropy(from_logits=True),
    metrics=['accuracy']
)

model.summary()

In [ ]:
history = model.fit(train_ds, epochs=10, 
validation_data=val_ds, 
validation_steps=30) 

In [ ]:
def plot_graphs(history, metric): 
plt.plot(history.history[metric]) 
plt.plot(history.history['val_'+metric], '') 
plt.xlabel("Epochs") 
plt.ylabel(metric) 
plt.legend([metric, 'val_'+metric]) 
 
plt.figure(figsize=(10, 5)) 
plt.subplot(1, 2, 1) 
plot_graphs(history, 'accuracy') 
plt.ylim(None, 1) 
plt.subplot(1, 2, 2) 
plot_graphs(history, 'loss') 
plt.ylim(0, None) 
plt.show()

In [ ]:
test_loss, test_acc = model.evaluate(test_ds) 
 
print('Test Loss:', test_loss) 
print('Test Accuracy:', test_acc) 

In [ ]:
import numpy as np 
 
samples = np.array([ 
'The movie was awesome, wonderful and amazing.', 
"The Movies is the bad and waste of time." 
]) 
predictions = model.predict(samples) 
 
predictions 